# 🎓 Campus Recruitment Prediction & HR Analytics
### Machine Learning Internship Capstone Project — Zenecks Infotech
**Author:** Aryaman Srimali Jain  
**Role:** Machine Learning Intern  
**Target:** Predict Student Campus Placement Outcomes (`Placed` vs `Not Placed`) & Identify Key Driving Factors  

---

## 1. Project Overview & Business Problem
Higher education institutions and corporate recruitment teams need data-driven insights into the key determinants of student employability. While student GPA is traditionally emphasized, campus placement decisions in corporate recruitment often depend on a complex interaction between secondary schooling, undergraduate specialization, prior work experience, and MBA employability tests.

### Core Objectives:
1. **Exploratory Data Analysis (EDA):** Identify which academic milestones (10th, 12th, Degree, MBA) and extracurricular factors (work experience, stream specialization) have the highest correlation with placement.
2. **Predictive Modeling:** Train and cross-validate multiple supervised classifiers (Logistic Regression, Random Forest, Gradient Boosting, and Support Vector Machine).
3. **Feature Importance & Decision Thresholds:** Derive feature importance rankings to assist university placement cells in mentoring at-risk students before recruitment drives.

## 2. Environment Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Load campus recruitment dataset
data_path = '../data/Placement_Data.csv' if Path('../data/Placement_Data.csv').exists() else 'Placement_Data.csv'
df = pd.read_csv(data_path)
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 3. Dataset Summary & Preprocessing Inspection
Checking data types, missing values, and column semantics:

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nPlacement Status Distribution:")
print(df['status'].value_counts(normalize=True).round(3))

# Drop serial number and post-placement salary (target leak)
df_clean = df.drop(columns=['sl_no', 'salary'])
df_clean.info()

## 4. Exploratory Data Analysis (EDA)
Analyzing how academic scores vary between placed and unplaced candidates:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features = [('ssc_p', '10th Percentage (SSC)'), ('hsc_p', '12th Percentage (HSC)'),
            ('degree_p', 'Undergraduate Degree %'), ('mba_p', 'MBA Percentage')]

for ax, (col, title) in zip(axes.flatten(), features):
    placed = df_clean[df_clean['status'] == 'Placed'][col]
    not_placed = df_clean[df_clean['status'] == 'Not Placed'][col]
    ax.hist(placed, bins=15, alpha=0.6, color='blue', label=f'Placed (μ={placed.mean():.1f}%)')
    ax.hist(not_placed, bins=15, alpha=0.6, color='red', label=f'Not Placed (μ={not_placed.mean():.1f}%)')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Score (%)')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()

## 5. Feature Encoding & Train-Test Partitioning
Converting categorical text features into numerical representations and standardizing feature scales:

In [ ]:
X = df_clean.drop(columns=['status']).copy()
y = (df_clean['status'] == 'Placed').astype(int).values

cat_cols = ['gender', 'ssc_b', 'hsc_b', 'hsc_s', 'degree_t', 'workex', 'specialisation']
encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c])
    encoders[c] = le

feature_names = list(X.columns)

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(X.values, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Training set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")

## 6. Supervised Machine Learning Benchmark
Training and comparing Logistic Regression, Random Forest, Gradient Boosting, and SVM:

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=3, random_state=42),
    'Support Vector Machine': SVC(kernel='rbf', probability=True, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    X_tr = X_train_scaled if name in ['Logistic Regression', 'Support Vector Machine'] else X_train
    X_te = X_test_scaled if name in ['Logistic Regression', 'Support Vector Machine'] else X_test
    
    cv_res = cross_validate(model, X_tr, y_train, cv=cv, scoring=['accuracy', 'f1', 'roc_auc'])
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    
    results.append({
        'Model': name,
        '5-Fold CV Acc': f"{cv_res['test_accuracy'].mean():.3f}",
        'Test Accuracy': f"{accuracy_score(y_test, y_pred):.3f}",
        'Precision': f"{precision_score(y_test, y_pred):.3f}",
        'Recall': f"{recall_score(y_test, y_pred):.3f}",
        'F1-Score': f"{f1_score(y_test, y_pred):.3f}",
        'ROC-AUC': f"{roc_auc_score(y_test, y_prob):.3f}"
    })

benchmark_table = pd.DataFrame(results)
display(benchmark_table)

## 7. Model Evaluation & Error Analysis
Detailed confusion matrix and classification report for the primary Logistic Regression model:

In [ ]:
best_lr = models['Logistic Regression']
y_pred_lr = best_lr.predict(X_test_scaled)

print("Classification Report (Logistic Regression):")
print(classification_report(y_test, y_pred_lr, target_names=['Not Placed', 'Placed']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

## 8. Feature Importance Analysis: What Determines Student Placement?
Extracting Gini importance from the Random Forest model:

In [ ]:
rf = models['Random Forest']
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances.plot(kind='barh', color='#2563eb')
plt.title('Relative Feature Importance in Predicting Campus Placement', fontweight='bold')
plt.xlabel('Gini Importance')
plt.tight_layout()
plt.show()

## 9. Real-Time Candidate Scoring Demonstration
Testing the trained pipeline with a mock candidate profile:

In [ ]:
# Profile: High SSC (82%), High HSC (85%), Science Degree (78%), Work Exp (Yes)
sample_candidate = np.array([[1, 82.0, 0, 85.0, 0, 2, 78.0, 2, 1, 80.0, 0, 71.0]])
sample_scaled = scaler.transform(sample_candidate)
prob = best_lr.predict_proba(sample_scaled)[0][1]

print(f"Candidate Placement Probability: {prob * 100:.2f}%")
print(f"Verdict: {'PLACED' if prob >= 0.5 else 'NOT PLACED'}")

## 10. Key Insights & HR Recommendations
1. **Primary Determinants**: Secondary school percentage (`ssc_p`) and undergraduate degree score (`degree_p`) are the two most decisive academic predictors of campus recruitment success.
2. **Work Experience Premium**: Candidates with prior internship/work experience (`workex = Yes`) showed an **85%+** placement rate compared to only **59%** for those without experience.
3. **Specialization Impact**: Students specializing in **Marketing & Finance (`Mkt&Fin`)** experienced higher recruitment conversion compared to **Marketing & HR (`Mkt&HR`)**.
4. **Actionable Mentorship**: University placement officers can flag students with `< 60%` across secondary and undergrad levels early for targeted skill enhancement workshops before campus drive commencement.